# 04 · 합성 데이터 증강

Train split 이미지에서 객체 crop을 추출하여 합성 배경에 붙여넣는 방식으로 데이터를 증강합니다.

**모드 선택**:
- `procedural` — GPU 불필요, 빠름 (기본 권장)
- `diffusion` — GPU 필요, 고품질 배경 (Stable Diffusion 2.1-base)

## ① 파라미터

In [ ]:
from pathlib import Path

# ── 수정 가능한 파라미터 ────────────────────────────────────────────
REPO_DIR       = Path(globals().get('REPO_DIR',       '/content/Military_Object_Detection'))
WORKSPACE_ROOT = Path(globals().get('WORKSPACE_ROOT', '/content/drive/MyDrive/Military_Object_Detection'))
DATASET_YAML   = Path(globals().get('DATASET_YAML',
                       str(WORKSPACE_ROOT / 'data' / 'processed' / 'yolo_dataset' / 'dataset.yaml')))

MODE            = globals().get('MODE', 'procedural')  # 'procedural' | 'diffusion' | 'auto'
SYNTHETIC_COUNT = int(globals().get('SYNTHETIC_COUNT', 500))
IMAGE_SIZE      = int(globals().get('IMAGE_SIZE', 640))
SEED            = int(globals().get('SEED', 42))

# diffusion 전용 파라미터
DIFFUSION_MODEL_ID = globals().get('DIFFUSION_MODEL_ID', 'stabilityai/stable-diffusion-2-1-base')
DIFFUSION_STEPS    = int(globals().get('DIFFUSION_STEPS', 20))
ALLOW_FALLBACK     = bool(globals().get('ALLOW_FALLBACK', True))   # diffusion 실패 시 procedural로 폴백

# 출력 디렉토리 (None이면 자동 설정)
OUTPUT_DIR = globals().get('OUTPUT_DIR', None)
if OUTPUT_DIR is None:
    OUTPUT_DIR = str(WORKSPACE_ROOT / 'data' / 'processed' / f'augmented_{MODE}')
OUTPUT_DIR = Path(OUTPUT_DIR)
# ──────────────────────────────────────────────────────────────────

print(f'MODE            : {MODE}')
print(f'SYNTHETIC_COUNT : {SYNTHETIC_COUNT}')
print(f'IMAGE_SIZE      : {IMAGE_SIZE}')
print(f'OUTPUT_DIR      : {OUTPUT_DIR}')

## ② 환경 초기화 및 의존성 확인

In [ ]:
import sys, os
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)
os.environ['MAD_WORKSPACE_ROOT'] = str(WORKSPACE_ROOT)

from mad.colab_utils import setup_colab_env, check_gpu, check_dataset

setup_colab_env(REPO_DIR, WORKSPACE_ROOT)
gpu_info = check_gpu(require=False)
check_dataset(DATASET_YAML)

if MODE == 'diffusion':
    print()
    if not gpu_info.get('cuda') and not gpu_info.get('mps'):
        print('⚠️  diffusion 모드에는 GPU가 강력히 권장됩니다.')
        print('   GPU 없이 실행하면 이미지 1장당 수 분이 소요될 수 있습니다.')
        print('   ALLOW_FALLBACK=True이면 자동으로 procedural 모드로 전환됩니다.')
    try:
        import diffusers, transformers
        print(f'✅ diffusers {diffusers.__version__}, transformers {transformers.__version__}')
    except ImportError:
        print('❌ diffusion 의존성 미설치. 다음 셀을 실행하세요:')
        print('   !pip install -q -r requirements-augmentation.txt')

In [ ]:
# diffusion 모드 사용 시 아래 셀을 실행하여 의존성 설치
# (이미 설치된 경우 건너뛰어도 됩니다)
import sys, subprocess
# subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
#                 '-r', str(REPO_DIR / 'requirements-augmentation.txt')], check=True)
print('의존성 설치가 필요하면 위 주석을 해제하고 실행하세요.')

## ③ 합성 데이터 생성

In [ ]:
from mad.synthetic_augmentation import SyntheticConfig, generate_augmented_dataset

meta = generate_augmented_dataset(
    SyntheticConfig(
        dataset_yaml=DATASET_YAML,
        output_dir=OUTPUT_DIR,
        synthetic_count=SYNTHETIC_COUNT,
        image_size=IMAGE_SIZE,
        mode=MODE,
        diffusion_model_id=DIFFUSION_MODEL_ID,
        diffusion_steps=DIFFUSION_STEPS,
        allow_fallback=ALLOW_FALLBACK,
        fallback_mode='procedural',
        seed=SEED,
        workspace_root=WORKSPACE_ROOT,
    )
)

print('\n✅ 합성 데이터셋 생성 완료')
print(f"   effective_mode          : {meta['effective_mode']}")
print(f"   synthetic_count_created : {meta['synthetic_count_created']}")
print(f"   output_dataset_yaml     : {meta['output_dataset_yaml']}")
if meta.get('fallback_reason'):
    print(f"   fallback_reason         : {meta['fallback_reason']}")

## ④ 생성 결과 확인

In [ ]:
import json
from pathlib import Path

meta_path = OUTPUT_DIR / 'metadata.json'
if meta_path.exists():
    import pprint
    pprint.pprint(json.loads(meta_path.read_text(encoding='utf-8')))

# 생성된 dataset.yaml 내용 확인
aug_yaml_path = OUTPUT_DIR / 'dataset.yaml'
if aug_yaml_path.exists():
    print('\n--- 생성된 dataset.yaml ---')
    print(aug_yaml_path.read_text(encoding='utf-8'))

## 다음 단계

생성된 `dataset.yaml`을 사용하여 `notebooks/03_run_benchmark.ipynb`에서 다음과 같이 실행하세요:
```python
DATASET_YAML = OUTPUT_DIR / 'dataset.yaml'
```

## CLI 동등 명령어
```bash
!python scripts/run_synthetic_augmentation.py \\
  --dataset-yaml $MAD_WORKSPACE_ROOT/data/processed/yolo_dataset/dataset.yaml \\
  --output-dir $MAD_WORKSPACE_ROOT/data/processed/augmented_procedural \\
  --mode procedural \\
  --synthetic-count 500 \\
  --seed 42
```